# 02 · Parcellation: source resolution, then the lobes12 and r18 stores (CPU)

1. **Anatomy** of lobes12 and r18 (no EEG): each hemisphere's 34 Desikan-Killiany labels partitioned, no
   cross-hemisphere region, identical left / right definitions, every region one contiguous piece of cortex, correct
   hemispheric placement, non-empty source sets.
2. **Analytic source resolution** per atlas, both inverses, with the stores' exact fsaverage BEM and inverse settings,
   and the pre-specified `atlas_resolution` rule (`configs/final/selection.yaml`, unchanged). Its verdict is
   authoritative and is never altered: with the exact BEM, lobes12 is rule-recommended and r18 fails the LCMV
   own-region dominance criterion (0.778 < 0.80).
3. **The 68-ROI stores' own operators** (real data, no labels; the corrected LCMV store reads the verified original
   checkpoints), feature reliability, reconstruction check.
4. **Authorization, then the new stores.** An atlas is reconstructed only if its anatomical checks passed AND it is
   rule-eligible (lobes12) OR the author's post-result approval of an exploratory atlas is recorded (r18: amendment
   A1, `configs/final/amendments.yaml`; set `EXPLORATORY_R18_APPROVAL = 'A1'` below to record it). The build refuses
   otherwise (`build_stores.py build`). Each store is built from the ORIGINAL recordings by the unchanged pipeline
   (merged Desikan-Killiany labels, `mean_flip` over their own sources; never averages of 68-ROI signals or PLV),
   LCMV and wMNE, window modes B and A; validated, wMNE matched to LCMV, and each matched window by window to its
   68-ROI store (participants, labels, windows, split, bands, sampling, sensor features).
5. **QC** of every new store (category A stops here), then the resolution and reliability of the operators built.

The 68-ROI stores are only read. No analysis atlas is selected here. Resumable (participant checkpoints). CPU,
about 1.5 h per atlas for the builds plus 30-60 min of diagnostics.

**How to use:** *Runtime → Run all*. Allow Google Drive access when asked. If Colab disconnects, open the notebook
again and *Run all* again: finished work is kept on Google Drive and skipped. Everything is written to
`<outputs>/FINAL_v1/` on Drive (never to the temporary Colab disk); the validated graph stores and all earlier
results are only read.

In [ ]:
# ===== Setup: nothing to edit. Mounts Drive, gets the code, installs the pinned environment. =====
UPDATE_CODE = False   # leave False. True only to move the release code pin deliberately to the newest commit
                      # (<outputs>/FINAL_v1/CODE_COMMIT.txt; unfinished work is then recomputed).
BRANCH = 'claude/unified-final-framework'
REPO_URL = 'https://github.com/Samimj9/Source-space-EEG-graph-learning-for-Alzheimer-s-disease-using-multi-band-functional-connectivity.git'

import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')

def _run(cmd, cwd=None):  # shows output and stops "Run all" on failure
    p = subprocess.run(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(p.stdout)
    if p.returncode:
        raise RuntimeError(f'failed: {" ".join(cmd)}')

os.chdir('/content')
_run(['rm', '-rf', '/content/repo'])
_run(['git', 'clone', '-q', '-b', BRANCH, REPO_URL, '/content/repo'])
os.chdir('/content/repo')
_run([sys.executable, 'scripts/colab_setup.py'] + (['--update-code'] if UPDATE_CODE else []))
sys.path.insert(0, '/content/repo/src')   # the package, for this notebook's own cells

from adgraph.colab import sh
from adgraph.config import load_paths
PATHS = load_paths('configs/paths.yaml')
OUT = PATHS['processed_dir']
FINAL = OUT + '/FINAL_v1'
print('release outputs on Drive:', FINAL)

In [ ]:
import glob, json
import pandas as pd
from IPython.display import Image, display
pd.set_option('display.max_rows', 400, 'display.width', 250, 'display.max_columns', 40)

def latest(pattern):
    found = sorted(glob.glob(pattern))
    return found[-1] if found else None

## 1. Anatomical checks

In [ ]:
NEW_ATLASES = ["lobes12", "r18"]
ATLAS_OK = {}
for atlas in NEW_ATLASES:
    try:
        sh(f"python scripts/final.py atlas-check --atlas {atlas}")
        ATLAS_OK[atlas] = True
    except RuntimeError:
        ATLAS_OK[atlas] = False
print(ATLAS_OK)

## 2. Analytic source resolution and the rule's recommendation (stores' BEM; unchanged)

In [ ]:
sh("python scripts/final.py parcellation")

In [ ]:
d = latest(FINAL + '/parcellation/resolution_*')
display(pd.read_csv(d + '/resolution_by_atlas.csv').round(3))
rec = json.load(open(d + '/recommendation.json'))
print('rule-recommended atlas:', rec['recommended'], '|', rec['forward_model'])
for v in rec['verdicts']:
    print(v['atlas'], 'eligible' if v['eligible'] else 'NOT eligible', '; '.join(v['reasons']))
if 'approx' in rec['forward_model'] or 'sphere' in rec['forward_model']:
    raise RuntimeError("the stores' BEM was not found: an approximate head model cannot authorize a reconstruction")

## 3. The 68-ROI stores' own operators, feature reliability, reconstruction check (read-only)

In [ ]:
for inverse in ('lcmv', 'wmne'):
    sh(f"python scripts/final.py store-resolution --inverse {inverse} --atlas dk68")
    sh(f"python scripts/final.py reliability --inverse {inverse} --mode B")
    sh(f"python scripts/final.py reconstruct-check --inverse {inverse} --mode B")

## 4. Authorization, then the new stores from the original recordings

`EXPLORATORY_R18_APPROVAL`: the author's post-result approval of the exploratory r18 arm. Leave `None` to build only
the rule-eligible atlas; set `'A1'` to record amendment A1 (once, in `FINAL_v1/decisions/exploratory_r18.json`; the
rule verdict is copied into it unchanged). Builds are resumable; `validation.json`, `match.json` and
`atlas_match.json` are written once. New directories next to the 68-ROI stores (outside `FINAL_v1/`), e.g.
`D3_LCMV_XU/graphsLX-lobes12_v1_<hash>`, `graphsLX10-lobes12_v1_<hash>`, `D3_WMNE_XU/graphsWX-r18_v1_<hash>`.

In [ ]:
EXPLORATORY_R18_APPROVAL = None   # set to 'A1' to record the author's approval (amendment A1, 2026-10-19)
import shlex
if EXPLORATORY_R18_APPROVAL:
    sh(f"python scripts/final.py approve-exploratory --atlas r18 --amendment {shlex.quote(EXPLORATORY_R18_APPROVAL)}")
BUILD = []
for atlas in NEW_ATLASES:
    try:
        sh(f"python scripts/final.py authorize --atlas {atlas}")
        BUILD.append(atlas)
    except RuntimeError:
        print(f'{atlas}: not built (not authorized; see above)')
if not BUILD:
    raise RuntimeError('no atlas is authorized for reconstruction; see the messages above')
for atlas in BUILD:
    for inverse in ('lcmv', 'wmne'):
        sh(f"python scripts/build_stores.py build --inverse {inverse} --atlas {atlas}")
        sh(f"python scripts/build_stores.py validate --inverse {inverse} --atlas {atlas}")
    sh(f"python scripts/build_stores.py match --atlas {atlas}")
    for inverse in ('lcmv', 'wmne'):
        sh(f"python scripts/build_stores.py match-atlas --inverse {inverse} --atlas {atlas}")
        sh(f"python scripts/build_stores.py size --inverse {inverse} --atlas {atlas}")
    sh(f"python scripts/final.py stores --atlas {atlas}")   # all four stores verify (locked when QC opens them)

## 5. QC of the new stores, then the resolution and reliability of the operators actually built

In [ ]:
UNRESOLVED = []
from IPython.display import Markdown
for ATLAS in BUILD:
    for inverse in ('lcmv', 'wmne'):
        for mode in ('B', 'A'):
            try:
                sh(f"python scripts/final.py qc --inverse {inverse} --mode {mode} --atlas {ATLAS}")
            except RuntimeError:
                UNRESOLVED.append((ATLAS, inverse, mode))
    for d in sorted(glob.glob(FINAL + f'/qc/*_*_{ATLAS}/')):
        if os.path.exists(d + 'qc_report.md'):
            display(Markdown(open(d + 'qc_report.md').read()))
if UNRESOLVED:
    raise RuntimeError(f'category A findings need a decision for {UNRESOLVED}: see the reports above and '
                       'docs/final/qc.md; nothing proceeds until then')

In [ ]:
for atlas in BUILD:
    for inverse in ('lcmv', 'wmne'):
        sh(f"python scripts/final.py store-resolution --inverse {inverse} --atlas {atlas}")
        sh(f"python scripts/final.py reliability --inverse {inverse} --mode B --atlas {atlas}")
for inverse in ('lcmv', 'wmne'):
    for atlas in ['dk68'] + BUILD:
        d = latest(f"{FINAL}/parcellation/store_resolution_{inverse}_{atlas}_*")
        t = pd.read_csv(d + '/per_participant.csv')
        print(inverse, atlas, t[['operator_rank', 'redundant_signals', 'crosstalk_fraction_median',
                                 'rois_dominated_by_own_region', 'leakage_corr_abs_median']].median().round(3).to_dict())